In [367]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [368]:
class value:
    def __init__(self,data,_children=(), _op='',label=''):
        self.data=data
        self._prev=set(_children)
        self._op=_op
        self._backward = lambda: None
        self.grad=0.0
        self.label=label

    def __add__(self, other):
        other=other if isinstance(other,value) else value(other) # (a+1)for operation with non-value objects like integers
        result = value(self.data + other.data, (self, other), "+")

        def _backward(): #backprop logic for addition op 
            self.grad = result.grad
            other.grad = result.grad
        result._backward = _backward

        return result
    
    def __radd__(self, other): # to be able to do expressions like (2+a)
         return self+other

    def __mul__(self, other):
            other=other if isinstance(other,value) else value(other)#to be able to do expression like (a*2)
            result = value(self.data * other.data , (self,other),"*")

            def _backward():#backprop logic 
                self.grad = other.data * result.grad
                other.grad = self.data * result.grad
            result._backward = _backward

            return result
    
    def __rmul__(self, other):  # to be able to do expressions like (2*a)
         return self*other
    
    def tanh(self): # this is the activation function we are using in this illustration and we choose to directly define it
         x=self.data
         t=(math.exp(2*x)-1)/(math.exp(2*x)+1)
         out=value(t,(self,),'tanh')

         def _backward():
            self.grad = (1-t**2)*out.grad

         out._backward = _backward

         return out

    def backward(self):
         topo=[]
         visited=set()

         def build_topo(v):
              if v not in visited:
                   visited.add(v)
                   for child in v._prev:
                        build_topo(child)
                   topo.append(v)
         build_topo(self)
         
         self.grad=1.0
         for node in reversed(topo):
            node._backward()
              

    def __repr__(self):
            return f"value(Data = {self.data})"
    


    

In [369]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) 

    nodes, edges=trace(root)
    
    for n in nodes:
        
        dot.node(name=str(id(n)), label = "{ data %.4f | grad %.4f}" % (n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [370]:
# A neuron is made up of two parts linear part and activation function

#inputs
x1=value(2.0,label='x1')
x2=value(0.0,label='x2')

#weights
w1=value(-3.0,label='w1')
w2=value(1.0,label='w2')

#bias of neuron
b=value(6.8813735870,label='b')

#linear part of the neuron z = x1*w1 + x2*w2 + b
x1w1=x1*w1;x1w1.label='x1w1'
x2w2=x2*w2;x2w2.label='x2w2'
_z=x1w1+x2w2;_z.label='_z'
z= _z + b;z.label='z'

o=z.tanh() #Activation function tangent hyperbola f(x)= {(e^2x+1)/(e^2x-1)}
o.label='o'
#draw_dot(o) #take a note that this graph just shows one neuron

# Neural Network 

In [371]:
import random
from BackpropEngine import Value

class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Neuron(Module):

    def __init__(self, nin, nonlin=True): #initializing weights randomly acc to no. of input
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin

    def __call__(self, x): # Forward pass calculation of a neuron
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

class Layer(Module): 

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

In [372]:
x=[2.0,3.0,-1.0]
n=MLP(3,[4,4,1])
n(x)
n(x).backward()
#draw_dot(n(x)) [output can be seen in image1.svg]

In [373]:
# example dataset test

xs=[
    [2.0,3.0,-1.0],
    [3.0,-1.0,0.5],
    [0.5,0.1,0.1],
    [1.0,1.0,-1.0]
]

ys=[1.0,-1.0,-1.0,1.0]#targets


#n.layers[0].neurons[0].w[0] # as soon we do loss.backward we have got a gradient for every neuron in every layer
#len(n.parameters()) #we have total 41 parameters 



In [374]:
ypred=[n(x) for x in xs]

loss=sum((yout - ygt)**2 for yout, ygt in zip(ys, ypred))# yout=output/prediction and ygt=ground truth

loss

Value(data=4.766628711377274, grad=0)

In [ ]:
#Execute the cell and below to see change of loss

ypred=[n(x) for x in xs]

loss=sum((yout - ygt)**2 for yout, ygt in zip(ys, ypred))# yout=output/prediction and ygt=ground truth

loss

Value(data=1.5138691918289822, grad=0)

In [400]:
loss.backward()

In [401]:
for p in n.parameters():
    p.data+= -0.01*p.grad # -0.01=alpha the learning rate and its negative because we want to minimize the loss

In [ ]:
#draw_dot(loss) [output in NN.svg]